# Colab Experiment Runner Template
Template for running CDGA and baseline semantic segmentation experiments on Google Colab.

In [ ]:
# ==============================================================================
# Cell 0: Setup Environment & Transfer Dataset
# ==============================================================================
import os
import shutil
import subprocess

# --- CONFIGURATION ---
DATASET = "vaihingen"      # Options: vaihingen, potsdamRGB, loveda
REPO_URL = "https://github.com/hoangnecon/cdga-experiments.git"
DRIVE_ROOT = "/content/drive/MyDrive/WACV_EXP"
PROJECT_DIR = "/content/cdga-experiments"
LOCAL_DATA_DIR = "/content/data"
# ---------------------

# 1. Clone/Update code repository
if not os.path.exists(PROJECT_DIR):
    print(f"Cloning repository to {PROJECT_DIR}...")
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    print(f"Updating repository at {PROJECT_DIR}...")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)

# 2. Install dependencies and setup environment
print("Running environment setup script...")
subprocess.run(["bash", f"{PROJECT_DIR}/setup_colab.sh"], check=True)

# 3. Extract or copy dataset to local VM SSD
print(f"Preparing dataset '{DATASET}' on local SSD...")
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
local_dataset_path = f"{LOCAL_DATA_DIR}/{DATASET}"
drive_dataset_tar = f"{DRIVE_ROOT}/data/{DATASET}" + ".tar"
drive_dataset_dir = f"{DRIVE_ROOT}/data/{DATASET}"

if os.path.exists(local_dataset_path):
    print(f"Dataset already exists at {local_dataset_path}")
else:
    if os.path.exists(drive_dataset_tar):
        print(f"Extracting {drive_dataset_tar} to local SSD...")
        subprocess.run(["tar", "-xf", drive_dataset_tar, "-C", LOCAL_DATA_DIR], check=True)
        print("Extraction complete.")
    elif os.path.exists(drive_dataset_dir):
        print(f"Copying {drive_dataset_dir} to local SSD...")
        shutil.copytree(drive_dataset_dir, local_dataset_path)
        print("Copy complete.")
        print("Creating tar archive on Google Drive for future runs...")
        try:
            subprocess.run(["tar", "-cf", drive_dataset_tar, "-C", LOCAL_DATA_DIR, DATASET], check=True)
            print("Archive created.")
        except Exception as e:
            print(f"Failed to create tar archive: {e}")
    else:
        raise FileNotFoundError(f"Dataset not found at {drive_dataset_tar} or {drive_dataset_dir}")

# Set environment variables
os.environ["PYTHONPATH"] = f"{PROJECT_DIR}:{PROJECT_DIR}/geoseg:" + os.environ.get("PYTHONPATH", "")
os.environ["GEOSEG_DIR"] = f"{PROJECT_DIR}/geoseg"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

print("Setup completed.")

In [ ]:
# ==============================================================================
# Cell 1: Verify Environment & Data Loading
# ==============================================================================
import os
import sys
import torch
from pathlib import Path

DATASET = "vaihingen"
METHOD = "cdga"
CONFIG_FILE = "experiments/configs/methods/gradient_based/cdga.yaml"
PROJECT_DIR = "/content/cdga-experiments"
LOCAL_DATA_DIR = f"/content/data/{DATASET}"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
if f"{PROJECT_DIR}/geoseg" not in sys.path:
    sys.path.insert(0, f"{PROJECT_DIR}/geoseg")

from shared.trainer import load_config
from shared.trainer import get_dataset_class

print("1. Loading dataset...")
dataset_cls = get_dataset_class(DATASET)
ds = dataset_cls(split="train", data_root=LOCAL_DATA_DIR)
print(f"Dataset loaded. Total samples: {len(ds)}")

print("2. Verifying configuration...")
config_path = Path(f"{PROJECT_DIR}/{CONFIG_FILE}")
cfg = load_config(config_path)
print(f"Config loaded. Dataset: {cfg['data']['dataset']}, Backbone: {cfg['model']['backbone']}")

print("3. Verification forward/backward pass...")
import importlib.util
method_dir_slug = "gradient_based" if "gradient" in CONFIG_FILE else "loss_based"
model_module_path = f"{PROJECT_DIR}/experiments/methods/group_a/{method_dir_slug}/{METHOD}/model.py"

spec = importlib.util.spec_from_file_location("model_module", model_module_path)
model_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_module)

model = model_module.build_model(cfg)
model.backbone = model.backbone.cuda()
model.train_mode()

sample = ds[0]
image = sample["image"].unsqueeze(0).cuda()
label = sample["label"].unsqueeze(0).cuda()
boundary_mask = sample.get("boundary_mask")
if boundary_mask is not None:
    boundary_mask = boundary_mask.unsqueeze(0).cuda()

loss = model.forward_train(image, label, boundary_mask=boundary_mask)
loss.backward()
print(f"Forward/backward pass verified. Loss: {loss.item():.4f}")

if hasattr(model, "_hook") and model._hook.grad_stats:
    stats = model._hook.grad_stats
    ratio = stats.get('mod_boundary', 0) / max(stats.get('orig_boundary', 1e-8), 1e-8)
    print(f"CDGA Amplification Ratio: {ratio:.2f}x")

print("All verification steps passed.")

In [ ]:
# ==============================================================================
# Cell 2: Run Training
# ==============================================================================
import os

DATASET = "vaihingen"
METHOD = "cdga"
CONFIG_NAME = "cdga.yaml"
BATCH_SIZE = 8
EPOCHS = 100

PROJECT_DIR = "/content/cdga-experiments"
LOCAL_DATA_DIR = f"/content/data/{DATASET}"
RUNS_DIR = "/content/drive/MyDrive/WACV_EXP/runs"
TIER = "gradient_based" if METHOD in ["cdga", "pcgrad"] else "loss_based"

TRAIN_SCRIPT = f"{PROJECT_DIR}/experiments/methods/group_a/{TIER}/{METHOD}/train.py"
CONFIG_PATH = f"{PROJECT_DIR}/experiments/configs/methods/{TIER}/{CONFIG_NAME}"

assert os.path.exists(TRAIN_SCRIPT), f"Train script not found: {TRAIN_SCRIPT}"
assert os.path.exists(CONFIG_PATH), f"Config file not found: {CONFIG_PATH}"

# Construct training command line arguments
cmd = f"""python "{TRAIN_SCRIPT}" \
    --config "{CONFIG_PATH}" \
    --override \
        data.data_root="{LOCAL_DATA_DIR}" \
        output.runs_dir="{RUNS_DIR}" \
        data.num_workers=2 \
        train.batch_size={BATCH_SIZE} \
        train.epochs={EPOCHS}
"""

print(f"Executing training command:\n{cmd}\n")
print("="*80 + "\n")

# Run in shell
!cd "{PROJECT_DIR}" && {cmd}